In [63]:
from langchain_openai import ChatOpenAI
import os

model = ChatOpenAI(
    model='Qwen/Qwen3-Omni-30B-A3B-Instruct',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [64]:
# 定义提示词模板
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

msg_key = "input"
prompt_template = ChatPromptTemplate.from_messages([
    ('system', '你是一个乐于助人的助手。用{language}尽可能回答所有问题'),
    MessagesPlaceholder(variable_name=msg_key)
])

In [65]:
# 定义链
chain = prompt_template | model

In [73]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

# 保存聊天记录   session_id : history
store = {}


def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


def clean_session(session_id):
    if session_id in store:
        store.pop(session_id)


# 聊天携带历史记录
chat_witch_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key=msg_key  # 每次聊天时候的发送msg的key
)

In [74]:
from langchain_core.messages import HumanMessage

clean_session('session1')
session_config = {
    "configurable": {
        'session_id': 'session1'
    }
}
# 第一轮对话
resp = chat_witch_history.invoke(
    {
        msg_key: [HumanMessage(content='我要吃粑粑')],
        "language": "zh-cn"
    },
    config=session_config)
resp.content

'我不太明白你说的"要吃粑粑"是什么意思。如果你是想表达想吃某种食物，比如"粑粑"这种食物，我可以告诉你，"粑粑"在某些方言里是一种用糯米做的小吃，类似糍粑或者粽子。如果你有其他需求或者想了解更多关于食物的信息，我很乐意帮助你。'

In [75]:
# 第二轮对话
resp = chat_witch_history.invoke(
    {
        msg_key: [HumanMessage(content='我就要吃粑粑')],
        "language": "zh-cn"
    },
    config=session_config)
resp.content

'我理解你可能在开玩笑，但还是想跟你认真说一说健康和安全方面的事情。如果你说的“吃粑粑”是指吃粪便的话，那非常危险，可能会导致严重疾病，如腹泻、寄生虫感染等。这是自然界中对人类健康起保护作用的常识，并非玩笑的话题。\n\n如果你其实是在用开玩笑的方式说话，那我们换个轻松的方式聊点别的吧！比如：你今天想不想听说个关于美食的小知识，比如中国的“打糍粑”是怎么做的？或者你有没有在过年回家时看见大人们在墙角堆的“粑粑”（其实是糖瓜）——那才是可以吃的传统美食哦！\n\n我在这里愿意陪你聊天，一起开心就行！😊'

In [77]:
[x.content for x in store['session1'].messages]

['我要吃粑粑',
 '我不太明白你说的"要吃粑粑"是什么意思。如果你是想表达想吃某种食物，比如"粑粑"这种食物，我可以告诉你，"粑粑"在某些方言里是一种用糯米做的小吃，类似糍粑或者粽子。如果你有其他需求或者想了解更多关于食物的信息，我很乐意帮助你。',
 '我就要吃粑粑',
 '我理解你可能在开玩笑，但还是想跟你认真说一说健康和安全方面的事情。如果你说的“吃粑粑”是指吃粪便的话，那非常危险，可能会导致严重疾病，如腹泻、寄生虫感染等。这是自然界中对人类健康起保护作用的常识，并非玩笑的话题。\n\n如果你其实是在用开玩笑的方式说话，那我们换个轻松的方式聊点别的吧！比如：你今天想不想听说个关于美食的小知识，比如中国的“打糍粑”是怎么做的？或者你有没有在过年回家时看见大人们在墙角堆的“粑粑”（其实是糖瓜）——那才是可以吃的传统美食哦！\n\n我在这里愿意陪你聊天，一起开心就行！😊']

In [78]:
# 第三轮会话 并使用流式输出

resp_stream = chat_witch_history.stream(
    {
        msg_key: [HumanMessage(content='那我想吃粑粑味的糍粑')],
        "language": "zh-cn"
    },
    config=session_config)
for chunk in resp_stream:
    # 一个chunk就是一个token
    print(chunk.content, end="-")

-哈哈-，-这么-一-说-我就-懂-了-！-你是-想-吃-粽子-、-糍-粑-之类的-糯米-点-心-，-但是-口味-要-“-粑-粑-味-”-是-吧-？-——-这种-“-味-”的-说法-其实-挺-有意思-，-不过-既然-你知道-“-粑-粑-”-其实是-食物-，-那-我们可以-来-认真-“-发挥-创意-”-一下-哦-！

-虽然-我们-不能-真的-让-糍-粑-变成-大自然-“-原-生态-”的-“-粑-粑-味-”，-但-我们可以-换个-思路-来-逗-你-一-乐-：

-👉- 想-象-一下-：
-🥲- 咸-清香-？-豆浆-味-？-  
-🌶- 带-点-辣-？-和-辣椒-粉-搓-搓-？-  
-🍯- 包-个-蜜-枣-，-甜-得-像-糖-霜-雪-糕-？-  
-🐸- 或-者-加-点-“-蛙-叫-”-奶油-，-出-个-丑-香-款-？

-（-别-担心-，-其实-根本-没有人-会-真的-想-吃-动物-排-泄-物-味道-……-）

-K-idding- kidding-！-🎉-  
-“-粑-粑-味-”-如果-作为-谐-音-梗-的话-，-其实是-“-不-必要-”的-“-不-”-代表-甜蜜-的-“-不是-”-；-“-粑-粑-”-两-字-念-起来-也-像-“-八-八-”，-谐-音-吉利-，在-一些-地方-还-寓意-“-发-发-”，-香-糯-有-前途-！

-所以-——-

-🔥-来-个-“-蓝色-的-粑-粑-味-”-麦-芽-糖-焗-糍-粑-，-外-皮-酥-的-头发-痒-的-，-内-里-软-糯-香-甜-的-幻想-风味-？-  
-🍵-或者-来-个-“-创新-”-口味-的-：-臭-鳜-鱼-味-？-（-笑着-）-  
-🤪-或者-干脆-摆-个-摊-：“-粑-粑-味-神-级-美食-限量-体验-版-——-试-吃-一次-，-健康成长-五年-！-”

-只要-别-吃-真的-自然界-产生的-那-东西-就行-，-哈哈哈哈-！

-总之-：-安心-吃-现-成-的-糯米-糕-、-糍-粑-、-豆-沙-包-云-吞-，-甜蜜-又-安全-！-要-关-照-自己的-胃-和-健康-哦-～---

In [79]:
# 新建一个会话
session_config= {
    "configurable": {
        'session_id': 'session2'
    }
}

resp_stream = chat_witch_history.stream(
    {
        msg_key: [HumanMessage(content='那我想吃粑粑味的糍粑')],
        "language": "zh-cn"
    },
    config=session_config)
for chunk in resp_stream:
    # 一个chunk就是一个token
    print(chunk.content, end="-")

-呵呵-，-这-可-真是-个-有趣-又-调皮-的想法-！-不过-“-粑-粑-味-”-可能-……-不太-能让-浅-浅-愉快-地-接受-哦-（-捂-嘴-笑-）-！-😄-

-其实-“-粑-粑-”-在-方言-里-是-“-便-便-”的-意思-，“-糍-粑-”-是一种-用-糯米-制成-的传统-美食-，-口感-软-糯-香-甜-。-你说-“-粑-粑-味-的-糍-粑-”，-大概-是在-开玩笑-，-或者是-挑战-一下-美食-的-创意-边界-？-🤔-

-要-不-咱-来-点-更有-意思-的-——-

-比如-，-你可以-试试-这些-口味-的-糍-粑-，-说不定-能-“-甜-得-你-笑-出-声-”，-也能-“-馋-到-你-忘记-所谓的-‘-粑-粑-味-’-”-：

--- 板-栗-蜜-味-糍-粑-：-香-甜-软-糯-，-像-秋天-一样-温暖-  
--- 红-糖-桂花-糍-粑-：-糯-叽-叽-中-带着-花-香-，-温柔-爆-棚-  
--- 玫-瑰-芝麻-糍-粑-：-颜值-和-口感-双双-在线-  
--- 咸-蛋-黄-肉-松-糍-粑-：-咸-甜-搭配-，-让人-欲-罢-不能-

-如果你-是在-玩-梗-或者-考-考-我的-脑-洞-……-那么-答案-揭晓-——-  
-👩-‍-🍳- 实-际-上-，-现在-有-好-多种-“-创意-口味-”的-糍-粑-，-比如-：

--- “-臭-豆腐-味-糍-粑-”-（-对不起-，-浅-浅-接受-不了-）-  
--- “-榴-莲-味-糍-粑-”-（-挑战-你的-味-蕾-极限-）-  
--- “-螺丝-粉-味-糍-粑-”-（-救-救-，-下-饭-神-搭配-！-）-  
--- “-奥-利-奥-可-可-味-糍-粑-”-（-我不-骗-你-，-真的-有-这-味道-！-）

-😉- 所-以-嘛-……-你想-吃-点-啥-口味-的-，-还是-故意-搞笑-一下-？-不管-怎样-，-浅-浅-都-陪你-玩-，-也-请你-相信-，-忠-于-美食-的-初心-——-甜-的-，-才是-最好-吃的-！-✨-

-（-悄悄-说-：-虽然-粑-粑-味-的-糍-粑-不太-现实-，-但-……-下次-再-想象-“-一种-甜-到-发光-，-却-尝-起来-像-菠-萝-脆-皮-的小-吃-”，-我-倒-是很-乐意-为你-定制-哦-～-）-🍍-✨---